# Train the Sentinel-1 flood segmentation U-Net (Phase 1, model C)

Runs on a free Colab T4. Pulls the Sen1Floods11 mirror tar (cc-by-4.0),
extracts only the India event's `S1Hand` + `LabelHand` pairs (~1.5 GB),
trains the U-Net with `ml/sar/train_unet.py`, reports an honest held-out val
IoU/Dice, and zips the artifacts for download into the repo's
`ml/artifacts/sar_unet/`.

**Setup:** put the repo's `ml/sar/` and `ml/requirements-ml.txt` on the Colab
runtime — either `git clone` (if you have a remote) or upload the `ml` folder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install torch segmentation-models-pytorch rasterio huggingface_hub

In [ ]:
%cd /content
# Either clone your remote or upload the ml folder. Example for a local upload:
import os
if not os.path.exists('/content/ml/sar/train_unet.py'):
    print('Upload the ml folder (File > Upload) then re-run this cell.')
    print('Or: !git clone <your-repo-url> && cd <repo>')

In [ ]:
# 1) Download the mirror tar (~35 GB, Google-side) + extract ONLY the India
#    event S1Hand/LabelHand pairs (~1.5 GB). The tar is deleted afterwards so
#    peak disk use is ~40 GB - fits the free T4's ~78 GB. Sen1Floods11 has a
#    single India event (2016 Assam, 535 chips), so this IS 'India only'.
!python ml/sar/download_sen1floods11.py --out /content/sen1floods11 --events India --layers S1Hand LabelHand
!df -h /content | tail -1

In [ ]:
# 2) Sanity: show tile / label pairs available.
tifs = !ls /content/sen1floods11/TP*/S1Hand_*.tif | head -20
tifs

In [ ]:
# 3) Train (T4: ~40-80 s/epoch @256). Quick-fit with --epochs 3 to confirm
#    the pipeline, then the real run with --epochs 35 (~25-45 min).
#    train/val are split deterministically by chip-id hash (85/15), so the
#    reported val IoU is on held-out chips (no leak).
!python ml/sar/train_unet.py --mode sen1floods11 --data-dir /content/sen1floods11 \
    --epochs 3 --size 256 --batch-size 8 --encoder resnet18 --out /content/artifacts/sar_unet
# Real run - re-run with a higher epoch count:
# !python ml/sar/train_unet.py --mode sen1floods11 --data-dir /content/sen1floods11 \
#     --epochs 35 --size 256 --batch-size 8 --encoder resnet18 --out /content/artifacts/sar_unet

In [ ]:
# 4) Report the measured numbers (this is what goes in the evidence sheet).
import json
meta = json.load(open('/content/artifacts/sar_unet/meta.json'))
print(f"val IoU = {meta['val_iou']:.4f}   val Dice = {meta['val_dice']:.4f}")
print(meta['note'])

In [ ]:
# 5) Download the artifacts, then drop them into ml/artifacts/sar_unet/ in
#    the repo (model.pt + meta.json). The backend picks them up automatically.
import shutil
shutil.make_archive('/content/sar_unet', 'zip', '/content/artifacts/sar_unet')
from google.colab import files
files.download('/content/sar_unet.zip')